# Vietnamese TTS - Google Colab UI
Chạy các cell bên dưới để khởi động giao diện Gradio cho TTS Engine.

Đảm bảo bạn đã upload hoặc clone source code `tts` và đang đứng tại thư mục gốc của project (có chứa `tts_engine.py`, `app.py`...).

In [ ]:
!pip install gradio edge-tts piper-tts requests

In [ ]:
import gradio as gr
import json
import os
import zipfile
import time
import datetime
import requests

# Import the local tts engine modules
try:
    import tts_engine
    import edge_tts_engine
    import zalo_tts_engine
except ImportError:
    print("Lỗi: Không tìm thấy các module TTS. Vui lòng đảm bảo bạn đang ở thư mục chứa mã nguồn của project.")

try:
    engine = tts_engine.TTSEngine()
    edge_engine = edge_tts_engine.EdgeTTSEngine()
except Exception as e:
    print("Error initializing engines:", e)
    engine = None
    edge_engine = None

# Load voices
try:
    voices = tts_engine.list_all_voices()
    voice_names = [v["name"] for v in voices]
    voice_map = {v["name"]: v for v in voices}
except Exception as e:
    print("Error loading voices:", e)
    voices = []
    voice_names = ["Default"]
    voice_map = {}

def process_tts(input_text, voice_name, speed, volume, noise, noise_w, pause, auto_download):
    if not input_text.strip():
        return None, "Vui lòng nhập nội dung văn bản hoặc JSON."

    output_dir = "colab_output"
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Check if input is JSON
    is_batch = False
    items = []
    try:
        data = json.loads(input_text)
        if isinstance(data, list) and len(data) > 0 and "text" in data[0]:
            is_batch = True
            items = data
    except json.JSONDecodeError:
        pass
    
    if not is_batch:
        items = [{"filename": f"output_{timestamp}.wav", "text": input_text.strip()}]
        
    voice = voice_map.get(voice_name)
    if not voice:
        return None, "Voice không hợp lệ."
        
    voice_type = voice.get("type", "")
    generated_files = []
    
    for item in items:
        text = item.get("text", "").strip()
        filename = item.get("filename", f"output_{time.time()}.wav")
        if not text:
            continue
            
        # Ensure filename ends with .wav
        if not filename.endswith(".wav"):
            filename += ".wav"
            
        filepath = os.path.join(output_dir, filename)
        
        try:
            if voice_type == "piper" and engine:
                audio, sr = engine.synthesize_piper(
                    text=text,
                    voice_path=voice["path"],
                    speed=speed,
                    volume=volume,
                    noise_scale=noise,
                    noise_w=noise_w,
                    pause=pause,
                )
                wav_bytes = engine.audio_to_wav(audio, sr)
                with open(filepath, "wb") as f:
                    f.write(wav_bytes)
                generated_files.append(filepath)
                
            elif voice_type == "edge" and edge_engine:
                wav_bytes = edge_engine.synthesize(
                    edge_voice=voice["edge_name"],
                    text=text,
                    speed=speed,
                    volume=volume,
                )
                with open(filepath, "wb") as f:
                    f.write(wav_bytes)
                generated_files.append(filepath)
                
            elif voice_type == "zalo":
                audio_url = zalo_tts_engine.synthesize(
                    speaker_id=voice["speaker_id"],
                    text=text,
                    speed=speed,
                )
                resp = requests.get(audio_url)
                if resp.status_code == 200:
                    with open(filepath, "wb") as f:
                        f.write(resp.content)
                    generated_files.append(filepath)
                else:
                    print(f"Error downloading from Zalo TTS: {resp.status_code}")
        except Exception as e:
            print(f"Lỗi khi tạo file {filename}: {e}")
            
    if not generated_files:
        return None, "Không có file nào được tạo ra."
        
    if is_batch or len(generated_files) > 1:
        zip_path = os.path.join(output_dir, f"batch_output_{timestamp}.zip")
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            for f in generated_files:
                zipf.write(f, os.path.basename(f))
        final_output = zip_path
    else:
        final_output = generated_files[0]
        
    if auto_download:
        try:
            from google.colab import files
            files.download(final_output)
        except ImportError:
            print("google.colab not found. Auto-download is only supported in Colab environments.")
            pass
            
    return final_output, f"Hoàn tất. Đã tạo thành công {len(generated_files)} file."

def create_ui():
    with gr.Blocks(title="Vietnamese TTS - Colab UI") as app:
        gr.Markdown("# 🎙️ Vietnamese TTS Engine - Google Colab UI")
        gr.Markdown("Công cụ tổng hợp giọng nói hỗ trợ cả chế độ Văn bản thường và JSON (Batch Mode).")
        
        with gr.Row():
            with gr.Column(scale=2):
                input_text = gr.Textbox(
                    label="Nội dung (Text hoặc JSON)",
                    lines=12,
                    placeholder='''Nhập văn bản bình thường hoặc nhập JSON dạng:\n[\n  {\n    "filename": "audio-scene-1-name.wav",\n    "text": "script-scene-1_text"\n  },\n  {\n    "filename": "audio-scene-2-name.wav",\n    "text": "script-scene-2_text"\n  }\n]'''
                )
                auto_download = gr.Checkbox(label="Auto Download (Tự động tải xuống khi chạy xong)", value=True)
                
            with gr.Column(scale=1):
                voice_dropdown = gr.Dropdown(
                    choices=voice_names, 
                    value=voice_names[0] if voice_names else None, 
                    label="Giọng đọc (Voice)"
                )
                speed = gr.Slider(0.3, 2.0, value=1.0, step=0.1, label="Tốc độ (Speed)")
                volume = gr.Slider(0.0, 1.5, value=1.0, step=0.1, label="Âm lượng (Volume)")
                
                with gr.Accordion("Cài đặt nâng cao (Piper)", open=False):
                    noise = gr.Slider(0.0, 1.5, value=0.667, step=0.001, label="Noise")
                    noise_w = gr.Slider(0.0, 2.0, value=0.8, step=0.1, label="Noise W")
                    pause = gr.Slider(0.0, 1.0, value=0.3, step=0.1, label="Pause")
                    
        btn_generate = gr.Button("🚀 Generate TTS", variant="primary")
        
        with gr.Row():
            output_file = gr.File(label="Tệp kết quả (WAV / ZIP)")
            output_msg = gr.Textbox(label="Trạng thái", interactive=False)
            
        btn_generate.click(
            fn=process_tts,
            inputs=[input_text, voice_dropdown, speed, volume, noise, noise_w, pause, auto_download],
            outputs=[output_file, output_msg]
        )
        
    return app

app = create_ui()
app.launch(debug=True, inline=True)
